# Notebook 03 - Treinamento do Modelo Pix2Pix

In [53]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import models

In [54]:
# Diretórios dos espectrogramas
input_dir_a = '../../dataset/spectrograms/Hall-Reverb/'
input_dir_b = '../../dataset/spectrograms/Chorus/'

In [55]:
class SpectrogramDataset(Dataset):
    def __init__(self, input_dir_a, input_dir_b):
        self.input_dir_a = input_dir_a
        self.input_dir_b = input_dir_b
        self.input_files = []
        self.output_files = []
        
        # Carregar todos os arquivos .npy
        for root_a, _, files_a in os.walk(input_dir_a):
            relative_path = os.path.relpath(root_a, input_dir_a)
            corresponding_dir_b = os.path.join(input_dir_b, relative_path)
            
            if not os.path.exists(corresponding_dir_b):
                continue
                
            files_a = sorted(f for f in files_a if f.endswith('.npy'))
            files_b = sorted(f for f in os.listdir(corresponding_dir_b) if f.endswith('.npy'))
            
            if len(files_a) != len(files_b):
                continue
            
            for file_a, file_b in zip(files_a, files_b):
                self.input_files.append(os.path.join(root_a, file_a))
                self.output_files.append(os.path.join(corresponding_dir_b, file_b))

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        input_file = np.load(self.input_files[idx])
        output_file = np.load(self.output_files[idx])
        
        # Converte de numpy para tensor
        input_tensor = torch.tensor(input_file, dtype=torch.float32)
        output_tensor = torch.tensor(output_file, dtype=torch.float32)
        
        # Adiciona a dimensão de batch
        input_tensor = input_tensor.unsqueeze(0)
        output_tensor = output_tensor.unsqueeze(0)
        
        # Remover a última dimensão caso seja extra
        input_tensor = input_tensor.squeeze(-1)
        output_tensor = output_tensor.squeeze(-1)

        return input_tensor, output_tensor
    
# Criar dataset e dataloader
train_dataset = SpectrogramDataset(input_dir_a, input_dir_b)
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)

print("Dados carregados com sucesso!")

Dados carregados com sucesso!


In [56]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        
        # Arquitetura do Gerador
        self.conv1 = nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.deconv1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1)
        
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = torch.relu(self.deconv1(x))
        x = torch.relu(self.deconv2(x))
        x = torch.tanh(self.deconv3(x))  # Saída normalizada para o intervalo [-1, 1]
        return x

In [57]:
import torch.nn.init as init

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        
        # Camadas convolucionais
        self.conv1 = nn.Conv2d(2, 64, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        
        # Camada totalmente conectada
        self.fc = nn.Linear(128 * 256 * 54, 1)

        # Inicialização de Xavier nos convolucionais
        init.xavier_uniform_(self.conv1.weight)
        init.xavier_uniform_(self.conv2.weight)
        
        # Inicialização de Xavier nos pesos da camada totalmente conectada (fc)
        init.xavier_uniform_(self.fc.weight)
        
        # Inicializar os vieses (bias) com valores pequenos
        init.zeros_(self.conv1.bias)
        init.zeros_(self.conv2.bias)
        init.zeros_(self.fc.bias)

    def forward(self, x, y):
        """
        Define a passagem do discriminador: concatenar a entrada x e a saída gerada y,
        passá-las pelas camadas convolucionais e depois pela camada totalmente conectada.
        """
        # Verificar se as imagens têm o mesmo tamanho, se não, redimensiona
        if x.shape[2:] != y.shape[2:]:
            y = F.interpolate(y, size=x.shape[2:], mode='bilinear', align_corners=False)
        
        # Concatenar a imagem real e a imagem gerada (x e y)
        combined = torch.cat((x, y), dim=1)  # Concatenar ao longo da dimensão dos canais (dim=1)

        # Passar pela primeira camada convolucional
        x = F.relu(self.conv1(combined))
        x = F.relu(self.conv2(x))

        # Achatar a saída para passar pela camada totalmente conectada (linear)
        x = torch.flatten(x, 1)  # Flatten, exceto para a dimensão do batch
        x = self.fc(x)  # Passar pela camada fc

        # Aplicar sigmóide para saída entre 0 e 1
        return torch.sigmoid(x)

In [58]:
# Verifique se CUDA está disponível e defina o dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# Mover o modelo para a GPU ou CPU
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Verifique o modelo
print(generator)
print(discriminator)


device: cuda
Generator(
  (conv1): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (conv3): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (deconv1): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (deconv2): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (deconv3): ConvTranspose2d(64, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
)
Discriminator(
  (conv1): Conv2d(2, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (fc): Linear(in_features=1769472, out_features=1, bias=True)
)


In [59]:
# Funções de perda
def discriminator_loss(real_output, fake_output):
    real_loss = nn.BCELoss()(real_output, torch.ones_like(real_output))
    fake_loss = nn.BCELoss()(fake_output, torch.zeros_like(fake_output))
    return real_loss + fake_loss

def generator_loss(fake_output, generated_image, target_image):
    # Redimensionar para o mesmo tamanho, caso necessário
    if generated_image.shape[2:] != target_image.shape[2:]:
        target_image = F.interpolate(target_image, size=generated_image.shape[2:], mode='bilinear', align_corners=False)

    gan_loss = nn.BCELoss()(fake_output, torch.ones_like(fake_output))
    l1_loss = nn.L1Loss()(generated_image, target_image)
    
    # Função de Perda Perceptual
    vgg = models.vgg16(pretrained=True).features.to(device).eval()
    def perceptual_loss(generated_image, target_image):
        # Se o número de canais for 1, duplicamos para 3 canais
        if generated_image.shape[1] == 1:
            generated_image = generated_image.repeat(1, 3, 1, 1)  # Replicar o canal para RGB
        if target_image.shape[1] == 1:
            target_image = target_image.repeat(1, 3, 1, 1)  # Replicar o canal para RGB

        generated_features = vgg(generated_image)
        target_features = vgg(target_image)
        
        return nn.MSELoss()(generated_features, target_features)

    
    perc_loss = perceptual_loss(generated_image, target_image)
    
    return gan_loss + 100 * l1_loss + 10 * perc_loss  # Ajuste os pesos conforme necessário


In [60]:
# Otimizadores
lr = 1e-5  # Taxa de aprendizado ajustada
generator_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))


In [61]:
# Função de treinamento
def train_step(input_image, target_image):
    input_image = input_image.to(device)
    target_image = target_image.to(device)
    
    # Treinamento do discriminador
    discriminator_optimizer.zero_grad()
    
    real_output = discriminator(input_image, target_image)
    fake_image = generator(input_image)
    fake_output = discriminator(input_image, fake_image.detach())
    
    # Perda do discriminador
    disc_loss = discriminator_loss(real_output, fake_output)
    disc_loss.backward()
    discriminator_optimizer.step()
    
    # Treinamento do gerador
    generator_optimizer.zero_grad()
    
    fake_output = discriminator(input_image, fake_image)
    gen_loss = generator_loss(fake_output, fake_image, target_image)
    gen_loss.backward()
    generator_optimizer.step()
    
    return gen_loss.item(), disc_loss.item()


In [ ]:
# Inicialização das listas para armazenar as perdas de cada época
gen_losses = []
disc_losses = []

epochs = 100

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    epoch_gen_loss = 0.0
    epoch_disc_loss = 0.0
    steps = 0
    
    for i, (input_image, target_image) in enumerate(train_dataloader):
        input_image = input_image.to(device)
        target_image = target_image.to(device)
        
        gen_loss, disc_loss = train_step(input_image, target_image)
        
        # Acumular perdas para a época
        epoch_gen_loss += gen_loss
        epoch_disc_loss += disc_loss
        
        steps += 1
        
        if i % 10 == 0:
            print(f"Step {i}, Gen Loss: {gen_loss}, Disc Loss: {disc_loss}")
    
    # Calcular a perda média por época
    epoch_gen_loss /= steps
    epoch_disc_loss /= steps
    gen_losses.append(epoch_gen_loss)
    disc_losses.append(epoch_disc_loss)
    
    os.makedirs("../../model/generator/", exist_ok=True)
    os.makedirs("../../model/discriminator/", exist_ok=True)
    
    torch.save(generator.state_dict(), f"../../model/generator/generator_epoch_{epoch+1}.pth")
    torch.save(discriminator.state_dict(), f"../../model/discriminator/discriminator_epoch_{epoch+1}.pth")
    
    print(f"Epoch {epoch+1} - Gen Loss: {epoch_gen_loss}, Disc Loss: {epoch_disc_loss}")


Epoch 1/100


Step 0, Gen Loss: 1805.0374755859375, Disc Loss: 1.5534757375717163
Step 10, Gen Loss: 1751.874755859375, Disc Loss: 0.002353674964979291
Step 20, Gen Loss: 1657.8291015625, Disc Loss: 0.0013800720917060971
Step 30, Gen Loss: 1720.8580322265625, Disc Loss: 0.0006143546779640019
Step 40, Gen Loss: 1723.2498779296875, Disc Loss: 0.00037140422500669956
Step 50, Gen Loss: 1572.782470703125, Disc Loss: 0.00047560560051351786
Step 60, Gen Loss: 1749.4036865234375, Disc Loss: 0.0001586613361723721
Step 70, Gen Loss: 1679.841552734375, Disc Loss: 0.0003372422361280769
Step 80, Gen Loss: 1724.1488037109375, Disc Loss: 0.00016536351176910102
Step 90, Gen Loss: 1749.781005859375, Disc Loss: 0.00013712517102248967
Step 100, Gen Loss: 1737.508544921875, Disc Loss: 0.00010298866254743189
Step 110, Gen Loss: 1678.987548828125, Disc Loss: 0.00011218633153475821
Step 120, Gen Loss: 1641.412109375, Disc Loss: 0.00023549670004285872
Step 130, Gen Loss: 1762.096923828125, Disc Loss: 9.325248538516462e-05


In [ ]:
# Plotar as perdas ao longo do treinamento
plt.plot(range(1, epochs+1), gen_losses, label='Loss do Gerador')
plt.plot(range(1, epochs+1), disc_losses, label='Loss do Discriminador')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.title('Evolução das Perdas')
plt.legend()
plt.grid(True)
plt.savefig("loss_plot_final.png")
plt.show()
